# Plant Leaf Health Classification (Healthy vs. Diseased)

CNN-based binary image classifier built during the AI Summer Training at  Ministry of Environment, Water and Agriculture.


In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

## 1. Load and extract the dataset

In [ ]:
with zipfile.ZipFile('/content/plant_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

for f in os.listdir('/content'):
    print(f)

In [ ]:
zip_path = '/content/plant_dataset.zip'
extract_path = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("extract all✅")

In [ ]:
base_dir = '/content/Data'  

print("📂branches of Data:")
print(os.listdir(base_dir))  # ['healthy', 'diseased']

In [ ]:
healthy_dir = '/content/Data/healthy'
diseased_dir = '/content/Data/diseased'

print("Healthy Samples: ", len(os.listdir(healthy_dir)))
print("Diseased Samples: ", len(os.listdir(diseased_dir)))

**Output:**
```
عدد الصور السليمة: 1478
عدد الصور المصابة: 997
```

## 2. Inspect a sample image

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

diseased_dir = '/content/Data/diseased'
img_path = os.path.join(diseased_dir, os.listdir(diseased_dir)[0])

img = mpimg.imread(img_path)
plt.imshow(img)
plt.title("dissesed leaf from dataset")
plt.axis('off')
plt.show()

## 3. Build data generators and train the CNN

In [ ]:
# الإعدادات العامة
img_size = 224
batch_size = 32
base_dir = '/content/Data'  # Healthy و Diseased

datagen = ImageDataGenerator(
    # Augmentation + Normalization
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

**Output:**
```
Found 1981 images belonging to 2 classes.
Found 494 images belonging to 2 classes.
```

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Healthy أو Diseased
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)

**Training log (20 epochs):**

| Epoch | Accuracy | Loss | Val Accuracy | Val Loss |
|---|---|---|---|---|
| 1 | 0.6798 | 0.9852 | 0.9008 | 0.2727 |
| 2 | 0.8923 | 0.2849 | 0.8583 | 0.4124 |
| 3 | 0.8992 | 0.3271 | 0.9433 | 0.1557 |
| 4 | 0.9373 | 0.1737 | 0.9656 | 0.1212 |
| 5 | 0.9691 | 0.0935 | 0.9636 | 0.1140 |
| 6 | 0.9687 | 0.1321 | 0.9696 | 0.1012 |
| 7 | 0.9705 | 0.1432 | 0.9656 | 0.0728 |
| 8 | 0.9745 | 0.1073 | 0.9737 | 0.0651 |
| 9 | 0.9719 | 0.0864 | 0.9858 | 0.0812 |
| 10 | 0.9819 | 0.0654 | 0.9757 | 0.0732 |
| 11 | 0.9899 | 0.0387 | 0.9777 | 0.0633 |
| 12 | 0.9951 | 0.0294 | 0.9838 | 0.0731 |
| 13 | 0.9936 | 0.0226 | 0.9757 | 0.1251 |
| 14 | 0.9946 | 0.0208 | 0.9737 | 0.1279 |
| 15 | 0.9793 | 0.0608 | 0.9676 | 0.0672 |
| 16 | 0.9799 | 0.0855 | 0.9838 | 0.0435 |
| 17 | 0.9920 | 0.0366 | 0.9919 | 0.0484 |
| 18 | 0.9934 | 0.0238 | 0.9899 | 0.0380 |
| 19 | 0.9961 | 0.0127 | 0.9960 | 0.0797 |
| 20 | 0.9999 | 0.0057 | 0.9818 | — |

## 4. Plot training vs. validation accuracy

In [ ]:
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')
plt.show()

## 5. Save the trained model

In [ ]:
model.save("leaf_classifier_224.h5")

In [ ]:
from google.colab import files
files.download("leaf_classifier_224.h5")

## 6. Test the model on new images

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

#  load_model
model = load_model("leaf_classifier_224.h5")


img_path = '/content/leaf.jpg'  
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # يصبح (1, 224, 224, 3)

# 
prediction = model.predict(img_array)[0][0]
label = "Healthy leaf" if prediction > 0.5 else "Diseased leaf"

# 
plt.imshow(img)
plt.title(f"Prediction: {label}")
plt.axis('off')
plt.show()

**Output:** `Prediction: Diseased leaf`

In [ ]:
model = load_model("leaf_classifier_224.h5")

img_path = '/content/test.image.jpg'  
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  #  (1, 224, 224, 3)

prediction = model.predict(img_array)[0][0]
label = "Healthy leaf" if prediction > 0.5 else "Diseased leaf"

plt.imshow(img)
plt.title(f"Prediction: {label}")
plt.axis('off')
plt.show()

**Output:** `Prediction: Diseased leaf`

In [ ]:
model = load_model("leaf_classifier_224.h5")

img_path = '/content/testt.jpg'
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)  #  (1, 224, 224, 3)

prediction = model.predict(img_array)[0][0]
label = "Healthy leaf" if prediction > 0.5 else "Diseased leaf"

plt.imshow(img)
plt.title(f"Prediction: {label}")
plt.axis('off')
plt.show()

**Output:** `Prediction: Healthy leaf (1.00)`